In [ ]:
from google.colab import drive
import os

# Thực hiện kết nối với Google Drive
drive.mount('/content/drive')

# Kiểm tra xem Drive đã được kết nối thành công chưa
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Mount Drive thành công!")
else:
    print("❌ Có lỗi xảy ra khi mount Drive.")

Mounted at /content/drive
✅ Mount Drive thành công!


In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

def get_img_array(img_path, size):
    # Đọc ảnh và tiền xử lý chuẩn DenseNet
    img = image.load_img(img_path, target_size=size)
    array = image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return tf.keras.applications.densenet.preprocess_input(array)

def make_gradcam_heatmap(img_array, model, last_conv_layer_name="relu"):
    # 1. Tạo một mô hình phụ lấy đầu ra từ lớp Conv cuối cùng
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # 2. Tính toán Gradient (Đạo hàm) của dự đoán so với lớp Conv cuối
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        # Vì đây là bài toán nhị phân (1 output), ta lấy thẳng giá trị preds
        class_channel = preds[:, 0]

    # 3. Lấy gradients
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # 4. Tính trung bình gradients theo không gian (Pooling) để lấy trọng số
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 5. Nhân các feature map với trọng số tương ứng
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # 6. Chuẩn hóa heatmap về [0, 1] để vẽ
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.4):
    # Đọc ảnh gốc
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))

    # Đổi heatmap thành ảnh màu RGB (dùng colormap JET của OpenCV)
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)

    # Chồng heatmap lên ảnh gốc
    superimposed_img = cv2.addWeighted(heatmap_colored, alpha, img, 1 - alpha, 0)

    # Vẽ kết quả
    fig = plt.figure(figsize=(10, 4))

    # NÂNG CẤP: Thêm tiêu đề lớn là tên file để dễ phân biệt khi in hàng loạt
    file_name = os.path.basename(img_path)
    fig.suptitle(f"Phân tích Grad-CAM: {file_name}", fontsize=14, fontweight='bold')

    plt.subplot(1, 3, 1)
    plt.title("Ảnh gốc (X-ray)")
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title("Bản đồ nhiệt (Heatmap)")
    plt.imshow(heatmap_resized, cmap='jet')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title("Kết quả chồng lớp (Overlay)")
    plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# ==========================================
# HÀM CHẠY HÀNG LOẠT (BATCH PROCESSING)
# ==========================================
def run_gradcam_batch(folder_path, model, num_images=10, last_conv_layer_name="relu"):
    """
    Quét thư mục, lấy ảnh và chạy Grad-CAM liên tục.
    - folder_path: Đường dẫn tới thư mục chứa ảnh X-ray.
    - num_images: Số lượng ảnh tối đa muốn in ra (có thể chỉnh được).
    """
    print(f"Đang tìm kiếm ảnh trong thư mục: {folder_path}...")

    # Tìm tất cả các file ảnh định dạng phổ biến (jpg, jpeg, png)
    image_paths = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
        # Hỗ trợ cả trường hợp viết hoa đuôi file (JPG, PNG)
        image_paths.extend(glob.glob(os.path.join(folder_path, ext.upper())))

    if len(image_paths) == 0:
        print("Không tìm thấy bức ảnh nào trong thư mục này. Vui lòng kiểm tra lại đường dẫn!")
        return

    # Giới hạn số lượng ảnh chạy theo yêu cầu
    image_paths = image_paths[:num_images]
    print(f"Bắt đầu vẽ Grad-CAM cho {len(image_paths)} ảnh...\n" + "="*50)

    for i, img_path in enumerate(image_paths):
        # print(f"[{i+1}/{len(image_paths)}] Đang xử lý: {os.path.basename(img_path)}")
        img_array = get_img_array(img_path, size=(224, 224))
        heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name=last_conv_layer_name)
        display_gradcam(img_path, heatmap)

# ==========================================
# CÁCH SỬ DỤNG
# ==========================================
# 1. Load mô hình của bạn
model_path = '/content/drive/MyDrive/AI_DATA/saved_models/DenseNet121_RadImageNet_Finetuned_best.h5'
print("Đang load mô hình, vui lòng đợi...")
model = load_model(model_path)

# 2. CHỈ ĐỊNH THƯ MỤC & CHẠY HÀNG LOẠT
# Thay vì trỏ vào 1 file cụ thể, hãy trỏ vào thư mục chứa các file đó
folder_test = '/content/drive/MyDrive/AI_DATA/demoVideos/2 ảnh ngoại kiểm'

# Điều chỉnh num_images thành số lượng bạn muốn in ra (ví dụ: 10)
run_gradcam_batch(folder_test, model, num_images=10, last_conv_layer_name="relu")

Đang load mô hình, vui lòng đợi...


Đang tìm kiếm ảnh trong thư mục: /content/drive/MyDrive/AI_DATA/demoVideos/2 ảnh ngoại kiểm...
Bắt đầu vẽ Grad-CAM cho 8 ảnh...


In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.densenet import preprocess_input
import zipfile

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN TÙY CHỈNH
# ==========================================
# 1. Đường dẫn file zip Dataset trên Google Drive
ZIP_PATH = "/content/drive/MyDrive/AI_DATA/Dataset.zip"

# 2. Nơi giải nén data tạm thời trên Colab (để đọc nhanh hơn)
DATA_EXTRACT_DIR = "/content/my_data"

# 3. Đường dẫn đến mô hình tốt nhất bạn đã train xong
MODEL_PATH = "/content/drive/MyDrive/AI_DATA/saved_models/DenseNet121_RadImageNet_Finetuned_best.h5"

# 4. Số lượng ảnh muốn bốc ngẫu nhiên để vẽ
NUM_IMAGES = 20

# ==========================================
# BƯỚC 1: GIẢI NÉN DỮ LIỆU (NẾU CHƯA CÓ)
# ==========================================
if not os.path.exists(DATA_EXTRACT_DIR):
    print(f"-> Đang giải nén tập dữ liệu từ {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_EXTRACT_DIR)
    print("-> Giải nén hoàn tất!")
else:
    print("-> Thư mục dữ liệu đã tồn tại, bỏ qua giải nén.")

# Tìm đường dẫn chính xác của thư mục test
TEST_DIR = os.path.join(DATA_EXTRACT_DIR, "test") # Cập nhật nếu cấu trúc zip của bạn khác

# ==========================================
# BƯỚC 2: TẢI MÔ HÌNH VÀ TÌM LỚP CONV CUỐI CÙNG
# ==========================================
print(f"-> Đang nạp mô hình từ: {MODEL_PATH}")
model = load_model(MODEL_PATH)

last_conv_layer_name = None
for layer in reversed(model.layers):
    try:
        # Sử dụng layer.output.shape (chuẩn Keras 3) thay vì layer.output_shape
        if len(layer.output.shape) == 4 and layer.name != 'input_1':
            last_conv_layer_name = layer.name
            break
    except Exception:
        # Bỏ qua nếu lớp đó không có thuộc tính shape (như Dropout, v.v.)
        continue

print(f"-> Lớp Convolutional cuối cùng được dùng cho Grad-CAM: {last_conv_layer_name}")
# ==========================================
# BƯỚC 3: HÀM TÍNH TOÁN GRAD-CAM (CHO MẠNG NHỊ PHÂN)
# ==========================================
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    # Tạo mô hình trung gian trả về [Đầu ra lớp Conv cuối, Đầu ra phân loại]
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs],
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        # Vì là binary classification (1 nơ ron sigmoid), ta lấy trực tiếp giá trị này
        class_channel = preds[0, 0]

    # Đạo hàm của output dự đoán theo feature map của lớp conv cuối
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # Tính trung bình gradient (Global Average Pooling cho Gradients)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Nhân từng kênh của feature map với trọng số gradient tương ứng
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Chỉ giữ lại các giá trị có ảnh hưởng tích cực (ReLU)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# ==========================================
# BƯỚC 4: HÀM CHỒNG HEATMAP LÊN ẢNH GỐC
# ==========================================
def overlay_gradcam(img_path, heatmap, alpha=0.5, colormap=cv2.COLORMAP_JET):
    # Đọc ảnh gốc bằng OpenCV
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))

    # Chuyển heatmap về [0, 255] và đổi kích thước bằng ảnh gốc
    heatmap = np.uint8(255 * heatmap)
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Phủ màu cho heatmap
    heatmap_colored = cv2.applyColorMap(heatmap_resized, colormap)

    # Chồng ảnh
    superimposed_img = cv2.addWeighted(heatmap_colored, alpha, img, 1 - alpha, 0)

    # Đổi BGR (OpenCV) sang RGB (Matplotlib) để in ra không bị sai màu
    return cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)

# ==========================================
# BƯỚC 5: CHỌN ẢNH NGẪU NHIÊN VÀ IN HÀNG LOẠT
# ==========================================
# Lấy toàn bộ đường dẫn ảnh từ thư mục test (Bỏ qua COVID19)
image_paths = []
for root, dirs, files in os.walk(TEST_DIR):
    # Lấy tên thư mục hiện tại đang quét
    current_folder = os.path.basename(root)

    # BỘ LỌC: CHỈ CHO PHÉP 2 THƯ MỤC NÀY ĐI QUA
    if current_folder not in ['NORMAL', 'PNEUMONIA']:
        continue

    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

# Kiểm tra xem có đủ ảnh để bốc không
if len(image_paths) < NUM_IMAGES:
    print(f"Cảnh báo: Chỉ tìm thấy {len(image_paths)} ảnh, ít hơn {NUM_IMAGES} ảnh yêu cầu.")
    NUM_IMAGES = len(image_paths)

# Bốc ngẫu nhiên X ảnh
selected_paths = random.sample(image_paths, NUM_IMAGES)

# Setup lưới hiển thị Matplotlib
cols = 4
rows = (NUM_IMAGES + cols - 1) // cols
plt.figure(figsize=(15, 4 * rows))

for i, img_path in enumerate(selected_paths):
    # Lấy nhãn thực tế
    true_label = os.path.basename(os.path.dirname(img_path))

    # Chuẩn bị ảnh để đưa vào model
    img = load_img(img_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array_preprocessed = preprocess_input(img_array)

    # 1. AI Dự đoán
    pred_prob = model.predict(img_array_preprocessed, verbose=0)[0][0]
    pred_label = "PNEUMONIA" if pred_prob > 0.5 else "NORMAL"

    # Đổi màu tiêu đề: Đúng -> Xanh lá, Sai -> Đỏ
    title_color = 'green' if pred_label == true_label else 'red'

    # 2. Tạo Heatmap
    heatmap = make_gradcam_heatmap(img_array_preprocessed, model, last_conv_layer_name)

    # 3. Chồng Heatmap lên ảnh
    cam_img = overlay_gradcam(img_path, heatmap)

    # 4. Đưa lên lưới (Grid)
    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(cam_img)
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({pred_prob:.2f})", color=title_color, fontsize=11, fontweight='bold')
    ax.axis("off")

plt.tight_layout()
plt.show()
print("-> HOÀN TẤT VẼ GRAD-CAM!")

-> Thư mục dữ liệu đã tồn tại, bỏ qua giải nén.
-> Đang nạp mô hình từ: /content/drive/MyDrive/AI_DATA/saved_models/DenseNet121_RadImageNet_Finetuned_best.h5


-> Lớp Convolutional cuối cùng được dùng cho Grad-CAM: relu


-> HOÀN TẤT VẼ GRAD-CAM!


train u-net


In [ ]:
import os
import glob
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# ==========================================
# 1. KẾT NỐI VÀ TẢI DỮ LIỆU TỰ ĐỘNG BẰNG KAGGLEHUB
# ==========================================
print("-> Đang cài đặt thư viện kagglehub (nếu chưa có)...")
!pip install -q kagglehub

import kagglehub

print("\n-> Đang tải bộ dữ liệu từ Kaggle thông qua kagglehub...")
# Lệnh này sẽ tự động tải file zip, giải nén và lưu vào thư mục cache ẩn của Colab
DATASET_PATH = kagglehub.dataset_download("nikhilpandey360/chest-xray-masks-and-labels")

print(f"-> Đã tải dữ liệu thành công! Đường dẫn gốc: {DATASET_PATH}\n")

# ==========================================
# 2. XỬ LÝ ĐƯỜNG DẪN ẢNH VÀ MẶT NẠ (MASKS)
# ==========================================
# Đường dẫn mới sẽ dựa trên DATASET_PATH
IMG_DIR = os.path.join(DATASET_PATH, 'Lung Segmentation', 'CXR_png')
MASK_DIR = os.path.join(DATASET_PATH, 'Lung Segmentation', 'masks')

image_paths = sorted(glob.glob(os.path.join(IMG_DIR, '*.png')))
mask_paths = sorted(glob.glob(os.path.join(MASK_DIR, '*.png')))

print(f"Tổng số ảnh gốc tìm thấy: {len(image_paths)}")
print(f"Tổng số ảnh mặt nạ tìm thấy: {len(mask_paths)}")

valid_images = []
valid_masks = []

for img_path in image_paths:
    base_name = os.path.basename(img_path).replace('.png', '')
    mask_match = [m for m in mask_paths if base_name in os.path.basename(m)]
    if mask_match:
        valid_images.append(img_path)
        valid_masks.append(mask_match[0])

print(f"-> Đã ghép nối thành công {len(valid_images)} cặp ảnh/mặt nạ.")

# ==========================================
# 3. HÀM DATA GENERATOR CHUẨN U-NET (TIẾT KIỆM RAM)
# ==========================================
IMG_SIZE = 256

def read_image_and_mask(img_path, mask_path):
    # Đọc và chuẩn hóa ảnh gốc (Đưa về thang xám 1 kênh)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=-1)

    # Đọc và chuẩn hóa mặt nạ (0 là nền, 1 là phổi)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = mask / 255.0
    mask = (mask > 0.5).astype(np.float32)
    mask = np.expand_dims(mask, axis=-1)

    return img, mask

# Chia tập Train/Val
X_train_paths, X_val_paths, y_train_paths, y_val_paths = train_test_split(
    valid_images, valid_masks, test_size=0.15, random_state=42
)

def data_generator(img_paths, mask_paths, batch_size=8):
    while True:
        for i in range(0, len(img_paths), batch_size):
            batch_imgs = img_paths[i:i+batch_size]
            batch_masks = mask_paths[i:i+batch_size]

            X, Y = [], []
            for img_p, mask_p in zip(batch_imgs, batch_masks):
                img, mask = read_image_and_mask(img_p, mask_p)
                X.append(img)
                Y.append(mask)

            yield np.array(X), np.array(Y)

# ==========================================
# 4. KIẾN TRÚC MẠNG U-NET & DICE LOSS
# ==========================================
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def dice_bce_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1.0 - dice_coef(y_true, y_pred)
    return bce + dice_loss

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 1)):
    inputs = Input(input_shape)

    # ENCODER
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = BatchNormalization()(c1)
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = BatchNormalization()(c2)
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = BatchNormalization()(c3)
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    p3 = MaxPooling2D((2, 2))(c3)

    # BOTTLENECK
    c4 = Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    c4 = BatchNormalization()(c4)
    c4 = Conv2D(256, (3, 3), activation='relu', padding='same')(c4)

    # DECODER
    u5 = UpSampling2D((2, 2))(c4)
    u5 = concatenate([u5, c3])
    c5 = Conv2D(128, (3, 3), activation='relu', padding='same')(u5)
    c5 = Conv2D(128, (3, 3), activation='relu', padding='same')(c5)

    u6 = UpSampling2D((2, 2))(c5)
    u6 = concatenate([u6, c2])
    c6 = Conv2D(64, (3, 3), activation='relu', padding='same')(u6)
    c6 = Conv2D(64, (3, 3), activation='relu', padding='same')(c6)

    u7 = UpSampling2D((2, 2))(c6)
    u7 = concatenate([u7, c1])
    c7 = Conv2D(32, (3, 3), activation='relu', padding='same')(u7)
    c7 = Conv2D(32, (3, 3), activation='relu', padding='same')(c7)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c7)

    model = Model(inputs=[inputs], outputs=[outputs])
    model.compile(optimizer='adam', loss=dice_bce_loss, metrics=[dice_coef, 'accuracy'])
    return model

print("-> Đang khởi tạo U-Net...")
model = build_unet()

# ==========================================
# 5. HUẤN LUYỆN MÔ HÌNH VÀ LƯU VÀO GOOGLE DRIVE
# ==========================================
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# THIẾT LẬP NƠI LƯU (ĐẢM BẢO THƯ MỤC NÀY TỒN TẠI TRÊN DRIVE CỦA BẠN)
SAVE_DIR = '/content/drive/MyDrive/AI_DATA/'
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
SAVE_PATH = os.path.join(SAVE_DIR, 'unet_lung_extractor.h5')

callbacks = [
    ModelCheckpoint(SAVE_PATH, monitor='val_dice_coef', mode='max', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_dice_coef', mode='max', patience=3, factor=0.5, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor='val_dice_coef', mode='max', patience=8, restore_best_weights=True)
]

BATCH_SIZE = 8
train_steps = max(1, len(X_train_paths) // BATCH_SIZE)
val_steps = max(1, len(X_val_paths) // BATCH_SIZE)

print(f"\n{'='*50}")
print(" BẮT ĐẦU HUẤN LUYỆN U-NET")
print(f"{'='*50}\n")

history = model.fit(
    data_generator(X_train_paths, y_train_paths, BATCH_SIZE),
    steps_per_epoch=train_steps,
    validation_data=data_generator(X_val_paths, y_val_paths, BATCH_SIZE),
    validation_steps=val_steps,
    epochs=30,
    callbacks=callbacks
)

print("\n-> CHÚC MỪNG! ĐÃ TRAIN XONG U-NET VÀ LƯU TẠI GOOGLE DRIVE!")

-> Đang cài đặt thư viện kagglehub (nếu chưa có)...

-> Đang tải bộ dữ liệu từ Kaggle thông qua kagglehub...
Using Colab cache for faster access to the 'chest-xray-masks-and-labels' dataset.
-> Đã tải dữ liệu thành công! Đường dẫn gốc: /kaggle/input/chest-xray-masks-and-labels

Tổng số ảnh gốc tìm thấy: 800
Tổng số ảnh mặt nạ tìm thấy: 704
-> Đã ghép nối thành công 704 cặp ảnh/mặt nạ.
-> Đang khởi tạo U-Net...
Mounted at /content/drive

 BẮT ĐẦU HUẤN LUYỆN U-NET

Epoch 1/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8780 - dice_coef: 0.7025 - loss: 0.5706
Epoch 1: val_dice_coef improved from None to 0.41629, saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5



Epoch 1: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 184s 2s/step - accuracy: 0.9338 - dice_coef: 0.8343 - loss: 0.3375 - val_accuracy: 0.7321 - val_dice_coef: 0.4163 - val_loss: 1.1513 - learning_rate: 0.0010
Epoch 2/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9740 - dice_coef: 0.9341 - loss: 0.1432
Epoch 2: val_dice_coef did not improve from 0.41629
74/74 ━━━━━━━━━━━━━━━━━━━━ 146s 2s/step - accuracy: 0.9745 - dice_coef: 0.9363 - loss: 0.1401 - val_accuracy: 0.7428 - val_dice_coef: 0.0975 - val_loss: 1.6874 - learning_rate: 0.0010
Epoch 3/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9777 - dice_coef: 0.9439 - loss: 0.1228
Epoch 3: val_dice_coef did not improve from 0.41629
74/74 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.9774 - dice_coef: 0.9436 - loss: 0.1239 - val_accuracy: 0.7103 - val_dice_coef: 0.0911 - val_loss: 1.7492 - learning_rate: 0.0010
Epoch 4/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step 


Epoch 7: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 116s 2s/step - accuracy: 0.9819 - dice_coef: 0.9553 - loss: 0.0972 - val_accuracy: 0.8386 - val_dice_coef: 0.5037 - val_loss: 1.3493 - learning_rate: 5.0000e-04
Epoch 8/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9825 - dice_coef: 0.9566 - loss: 0.0936
Epoch 8: val_dice_coef improved from 0.50375 to 0.78608, saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5



Epoch 8: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.9825 - dice_coef: 0.9565 - loss: 0.0938 - val_accuracy: 0.9140 - val_dice_coef: 0.7861 - val_loss: 0.5947 - learning_rate: 5.0000e-04
Epoch 9/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9824 - dice_coef: 0.9571 - loss: 0.0942
Epoch 9: val_dice_coef improved from 0.78608 to 0.92655, saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5



Epoch 9: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 115s 2s/step - accuracy: 0.9827 - dice_coef: 0.9572 - loss: 0.0921 - val_accuracy: 0.9679 - val_dice_coef: 0.9266 - val_loss: 0.1796 - learning_rate: 5.0000e-04
Epoch 10/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9825 - dice_coef: 0.9580 - loss: 0.0928
Epoch 10: val_dice_coef improved from 0.92655 to 0.94793, saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5



Epoch 10: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 117s 2s/step - accuracy: 0.9830 - dice_coef: 0.9582 - loss: 0.0901 - val_accuracy: 0.9772 - val_dice_coef: 0.9479 - val_loss: 0.1379 - learning_rate: 5.0000e-04
Epoch 11/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9832 - dice_coef: 0.9590 - loss: 0.0886
Epoch 11: val_dice_coef improved from 0.94793 to 0.95063, saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5


... (2 outputs left out)


Epoch 18: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.9862 - dice_coef: 0.9663 - loss: 0.0708 - val_accuracy: 0.9783 - val_dice_coef: 0.9526 - val_loss: 0.1315 - learning_rate: 2.5000e-04
Epoch 19/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9861 - dice_coef: 0.9664 - loss: 0.0710
Epoch 19: val_dice_coef did not improve from 0.95255
74/74 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.9862 - dice_coef: 0.9665 - loss: 0.0704 - val_accuracy: 0.9767 - val_dice_coef: 0.9484 - val_loss: 0.1344 - learning_rate: 2.5000e-04
Epoch 20/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9867 - dice_coef: 0.9676 - loss: 0.0686
Epoch 20: val_dice_coef did not improve from 0.95255
74/74 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.9864 - dice_coef: 0.9671 - loss: 0.0697 - val_accuracy: 0.9782 - val_dice_coef: 0.9503 - val_loss: 0.1221 - learning_rate: 2.5000e-04
Epoch 21/30
74/74 ━━━━━━━━━━━━━━


Epoch 22: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.9875 - dice_coef: 0.9697 - loss: 0.0632 - val_accuracy: 0.9797 - val_dice_coef: 0.9549 - val_loss: 0.1196 - learning_rate: 1.2500e-04
Epoch 23/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9887 - dice_coef: 0.9718 - loss: 0.0576
Epoch 23: val_dice_coef improved from 0.95489 to 0.95512, saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5



Epoch 23: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.9883 - dice_coef: 0.9715 - loss: 0.0591 - val_accuracy: 0.9798 - val_dice_coef: 0.9551 - val_loss: 0.1210 - learning_rate: 1.2500e-04
Epoch 24/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9890 - dice_coef: 0.9728 - loss: 0.0560
Epoch 24: val_dice_coef did not improve from 0.95512
74/74 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.9888 - dice_coef: 0.9727 - loss: 0.0567 - val_accuracy: 0.9795 - val_dice_coef: 0.9551 - val_loss: 0.1290 - learning_rate: 1.2500e-04
Epoch 25/30
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9893 - dice_coef: 0.9739 - loss: 0.0540
Epoch 25: val_dice_coef did not improve from 0.95512
74/74 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.9891 - dice_coef: 0.9736 - loss: 0.0549 - val_accuracy: 0.9783 - val_dice_coef: 0.9531 - val_loss: 0.1427 - learning_rate: 1.2500e-04
Epoch 26/30
74/74 ━━━━━━━━━━━━━━


Epoch 30: finished saving model to /content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5
74/74 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.9903 - dice_coef: 0.9764 - loss: 0.0487 - val_accuracy: 0.9793 - val_dice_coef: 0.9563 - val_loss: 0.1385 - learning_rate: 3.1250e-05

-> CHÚC MỪNG! ĐÃ TRAIN XONG U-NET VÀ LƯU TẠI GOOGLE DRIVE!


Giải nén dữ liệu để crop phổi

In [ ]:
import zipfile
# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN GIẢI NÉN
# ==========================================
ZIP_PATH = "/content/drive/MyDrive/AI_DATA/Dataset.zip" # Đổi lại nếu tên file zip của bạn khác
RAW_DATA_DIR = "/content/my_data"

print(f"-> Đang kiểm tra dữ liệu tại {RAW_DATA_DIR}...")

if not os.path.exists(RAW_DATA_DIR):
    print(f"-> Bắt đầu giải nén từ {ZIP_PATH} (Sẽ mất khoảng vài chục giây)...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(RAW_DATA_DIR)
    print("-> Giải nén hoàn tất! Dữ liệu gốc đã sẵn sàng.")
else:
    print("-> Dữ liệu gốc đã tồn tại, bỏ qua bước giải nén.")

-> Đang kiểm tra dữ liệu tại /content/my_data...
-> Bắt đầu giải nén từ /content/drive/MyDrive/AI_DATA/Dataset.zip (Sẽ mất khoảng vài chục giây)...
-> Giải nén hoàn tất! Dữ liệu gốc đã sẵn sàng.


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from tensorflow.keras.models import load_model

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
# Nơi chứa ảnh gốc hiện tại của bạn
RAW_DATA_DIR = "/content/my_data"

# Nơi lưu ảnh đã được cắt nền (Masked Dataset)
MASKED_DATA_DIR = "/content/masked_data"

# Đường dẫn đến mô hình U-Net bạn vừa train
UNET_MODEL_PATH = "/content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5"

print("-> Đang tải U-Net (compile=False để tiết kiệm RAM)...")
# Dùng compile=False để tránh lỗi không nhận diện được hàm Dice Loss custom
unet_model = load_model(UNET_MODEL_PATH, compile=False)

print(f"-> Bắt đầu xử lý ảnh từ {RAW_DATA_DIR} sang {MASKED_DATA_DIR}...\n")

# Đếm tổng số ảnh để làm thanh tiến trình
all_img_paths = []
for root, _, files in os.walk(RAW_DATA_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            all_img_paths.append(os.path.join(root, file))

# Xử lý từng ảnh
for img_path in tqdm(all_img_paths, desc="Đang cắt nền phổi"):
    # Đọc ảnh gốc bằng OpenCV
    img = cv2.imread(img_path)
    if img is None: continue

    # Lưu lại kích thước gốc để không làm biến dạng ảnh
    original_shape = img.shape[:2]

    # 1. Chuẩn bị ảnh đầu vào cho U-Net (256x256, Grayscale)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_resized = cv2.resize(gray, (256, 256))
    gray_normalized = gray_resized / 255.0
    unet_input = np.expand_dims(gray_normalized, axis=(0, -1))

    # 2. U-Net dự đoán Mặt nạ
    mask = unet_model.predict(unet_input, verbose=0)[0]

    # 3. Phân ngưỡng và Trả mặt nạ về kích thước gốc
    mask_resized = cv2.resize(mask, (original_shape[1], original_shape[0]))
    binary_mask = (mask_resized > 0.5).astype(np.float32)

    # Nhân bản mask lên 3 kênh màu để khớp với ảnh RGB
    binary_mask_3c = np.stack((binary_mask,)*3, axis=-1)

    # 4. Áp mặt nạ (Nhân ma trận: Vùng ngoài phổi x 0 = Màu đen)
    masked_img = (img * binary_mask_3c).astype(np.uint8)

    # 5. Lưu ảnh sang thư mục mới, giữ nguyên cấu trúc thư mục con (train/test/val)
    relative_path = os.path.relpath(img_path, RAW_DATA_DIR)
    save_path = os.path.join(MASKED_DATA_DIR, relative_path)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    cv2.imwrite(save_path, masked_img)

print("\n-> XONG! Bạn đã có một Dataset hoàn toàn mới, vô nhiễm với hiệu ứng Clever Hans!")

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from tensorflow.keras.models import load_model

RAW_DATA_DIR = "/content/my_data"
MASKED_DATA_DIR = "/content/masked_data"
UNET_MODEL_PATH = "/content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5"

print("-> Đang tải mô hình U-Net...")
unet_model = load_model(UNET_MODEL_PATH, compile=False)

print(f"-> Bắt đầu xử lý ảnh từ {RAW_DATA_DIR} sang {MASKED_DATA_DIR}...\n")

# Lọc tìm danh sách ảnh cần cắt (Thông minh)
images_to_process = []
skipped_count = 0

for root, _, files in os.walk(RAW_DATA_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(root, file)

            # Tính toán trước đường dẫn file lưu
            relative_path = os.path.relpath(img_path, RAW_DATA_DIR)
            save_path = os.path.join(MASKED_DATA_DIR, relative_path)

            # --- KIỂM TRA ĐIỀU KIỆN CHẠY TIẾP ---
            if os.path.exists(save_path):
                skipped_count += 1
            else:
                images_to_process.append((img_path, save_path))

if skipped_count > 0:
    print(f"-> Đã tìm thấy {skipped_count} ảnh đã cắt từ trước. Sẽ tự động bỏ qua!")

if len(images_to_process) == 0:
    print("-> Tất cả dữ liệu đã được cắt xong! Không cần làm gì thêm.")
else:
    print(f"-> Cần cắt {len(images_to_process)} ảnh còn lại...\n")

    # Chỉ lặp qua những ảnh chưa cắt
    for img_path, save_path in tqdm(images_to_process, desc="Đang cắt nền phổi"):
        img = cv2.imread(img_path)
        if img is None: continue

        original_shape = img.shape[:2]

        # Tiền xử lý cho U-Net
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_resized = cv2.resize(gray, (256, 256))
        gray_normalized = gray_resized / 255.0
        unet_input = np.expand_dims(gray_normalized, axis=(0, -1))

        # Dự đoán
        mask = unet_model.predict(unet_input, verbose=0)[0]

        # Áp mặt nạ
        mask_resized = cv2.resize(mask, (original_shape[1], original_shape[0]))
        binary_mask = (mask_resized > 0.5).astype(np.float32)
        binary_mask_3c = np.stack((binary_mask,)*3, axis=-1)

        masked_img = (img * binary_mask_3c).astype(np.uint8)

        # Tạo thư mục con nếu chưa có và lưu file
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, masked_img)

    print("\n-> XONG! Đã cắt xong toàn bộ dataset.")

lọc + cắt ảnh còn lại


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from tensorflow.keras.models import load_model

RAW_DATA_DIR = "/content/my_data"
MASKED_DATA_DIR = "/content/masked_data"
UNET_MODEL_PATH = "/content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5"

print("-> Đang tải mô hình U-Net...")
unet_model = load_model(UNET_MODEL_PATH, compile=False)

print(f"-> Bắt đầu phân tích dữ liệu từ {RAW_DATA_DIR} sang {MASKED_DATA_DIR}...\n")

images_to_process = []
skipped_cut_count = 0   # Đếm số ảnh ĐÃ ĐƯỢC CẮT (Resume)
ignored_class_count = 0 # Đếm số ảnh rác KHÔNG PHẢI Normal/Pneumonia

for root, _, files in os.walk(RAW_DATA_DIR):
    current_folder = os.path.basename(root)

    # ==========================================
    # BỘ LỌC 1: CHỈ LẤY THƯ MỤC NORMAL & PNEUMONIA
    # ==========================================
    if current_folder not in ['NORMAL', 'PNEUMONIA']:
        ignored_class_count += sum(1 for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg')))
        continue

    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(root, file)
            relative_path = os.path.relpath(img_path, RAW_DATA_DIR)
            save_path = os.path.join(MASKED_DATA_DIR, relative_path)

            # ==========================================
            # BỘ LỌC 2: BỎ QUA NHỮNG FILE ĐÃ ĐƯỢC CẮT
            # ==========================================
            if os.path.exists(save_path):
                skipped_cut_count += 1
            else:
                images_to_process.append((img_path, save_path))

# In báo cáo rà soát trước khi chạy AI
print(f"-> Đã chủ động bỏ qua {ignored_class_count} ảnh rác (VD: COVID19).")

if skipped_cut_count > 0:
    print(f"-> Đã tìm thấy {skipped_cut_count} ảnh ĐÃ CẮT TỪ TRƯỚC. Sẽ tự động bỏ qua để tiết kiệm thời gian!")

# Quyết định chạy AI
if len(images_to_process) == 0:
    print("-> Tất cả dữ liệu cần thiết đều đã được cắt xong! Không cần chạy lại AI.")
else:
    print(f"-> Bắt đầu cắt {len(images_to_process)} ảnh mới...\n")

    for img_path, save_path in tqdm(images_to_process, desc="Đang cắt nền phổi"):
        img = cv2.imread(img_path)
        if img is None: continue

        original_shape = img.shape[:2]

        # Tiền xử lý cho U-Net
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray_resized = cv2.resize(gray, (256, 256))
        gray_normalized = gray_resized / 255.0
        unet_input = np.expand_dims(gray_normalized, axis=(0, -1))

        # U-Net Dự đoán
        mask = unet_model.predict(unet_input, verbose=0)[0]

        # Áp mặt nạ
        mask_resized = cv2.resize(mask, (original_shape[1], original_shape[0]))
        binary_mask = (mask_resized > 0.5).astype(np.float32)
        binary_mask_3c = np.stack((binary_mask,)*3, axis=-1)

        masked_img = (img * binary_mask_3c).astype(np.uint8)

        # Tạo thư mục và Lưu file
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, masked_img)

    print("\n-> HOÀN TẤT! Đã cập nhật đầy đủ dataset chuẩn.")

-> Đang tải mô hình U-Net...
-> Bắt đầu phân tích dữ liệu từ /content/my_data sang /content/masked_data...

-> Đã chủ động bỏ qua 576 ảnh rác (VD: COVID19).
-> Đã tìm thấy 2438 ảnh ĐÃ CẮT TỪ TRƯỚC. Sẽ tự động bỏ qua để tiết kiệm thời gian!
-> Bắt đầu cắt 3418 ảnh mới...



Đang cắt nền phổi: 100%|██████████| 3418/3418 [42:28<00:00,  1.34it/s]


-> HOÀN TẤT! Đã cập nhật đầy đủ dataset chuẩn.


train lại để lấy confusion matrix và training log csv

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Concatenate, BatchNormalization, LeakyReLU, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
import kagglehub
import gc

# ==========================================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================================
UNET_PATH = '/content/drive/MyDrive/AI_DATA/unet_lung_extractor.h5'
DENSENET_PATH = '/content/drive/MyDrive/AI_DATA/saved_models/DenseNet121_RadImageNet_MaskedData_Finetuned_best.h5'
LOCAL_MASKED_DIR = '/content/CheXpert_Masked_Images_v2'
LOCAL_CSV_PATH = '/content/masked_train.csv'
NEW_SAVE_PATH = '/content/drive/MyDrive/AI_DATA/multimodal_chexpert_best_V3(12 thang 5).h5'
HISTORY_CSV_PATH = '/content/drive/MyDrive/AI_DATA/training_history_v3.csv' # Nơi lưu nhật ký

os.makedirs(LOCAL_MASKED_DIR, exist_ok=True)
MAX_SAMPLES = 10000 # Đổi thành None nếu muốn train toàn bộ

# ==========================================================
# PHẦN 1: HÀM CẮT PHỔI & CHUẨN BỊ DATA
# ==========================================================
def extract_lungs(img_bgr, unet_model):
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_resized = cv2.resize(img_gray, (256, 256))
    img_input = np.expand_dims(img_resized / 255.0, axis=(0, -1))
    mask = unet_model.predict(img_input, verbose=0)[0]
    mask = (mask > 0.5).astype(np.uint8) * 255
    kernel_morph = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask_opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_morph)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_opened, connectivity=8)
    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        largest_labels = areas.argsort()[-2:][::-1] + 1
        clean_mask = np.zeros_like(mask_opened)
        for label in largest_labels:
            clean_mask[labels == label] = 255
    else:
        clean_mask = mask_opened
    clean_mask = cv2.dilate(clean_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(clean_mask)
    cv2.drawContours(filled_mask, contours, -1, 255, thickness=cv2.FILLED)
    final_mask = (filled_mask / 255.0).astype(np.float32)
    mask_resized = cv2.resize(final_mask, (img_bgr.shape[1], img_bgr.shape[0]))
    mask_3ch = np.stack((mask_resized,)*3, axis=-1)
    return (img_bgr * mask_3ch).astype(np.uint8)

def download_and_preprocess_data():
    print("📥 [1/3] Đang tải Dataset CheXpert từ Kaggle...")
    dataset_path = kagglehub.dataset_download("mimsadiislam/chexpert")
    csv_candidates = [os.path.join(root, name) for root, dirs, files in os.walk(dataset_path) for name in files if name == 'train.csv']
    if not csv_candidates: raise ValueError("❌ Không tìm thấy train.csv!")
    original_csv = csv_candidates[0]
    df = pd.read_csv(original_csv)
    df = df[['Path', 'Age', 'Sex', 'Pneumonia']]
    df = df[df['Pneumonia'].isin([0.0, 1.0])].copy()
    if MAX_SAMPLES:
        df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)
    print(f"\n✂️ [2/3] BẮT ĐẦU CẮT {len(df)} PHỔI BẰNG U-NET...")
    unet = load_model(UNET_PATH, compile=False)
    new_data = []
    for index, row in tqdm(df.iterrows(), total=len(df)):
        orig_img_path = os.path.join(dataset_path, row['Path'])
        safe_filename = row['Path'].replace('/', '_')
        new_img_path = os.path.join(LOCAL_MASKED_DIR, safe_filename)
        img = cv2.imread(orig_img_path)
        if img is not None:
            try:
                masked_img = extract_lungs(img, unet)
                cv2.imwrite(new_img_path, masked_img)
                new_data.append({'Masked_Path': new_img_path, 'Age': row['Age'], 'Sex': row['Sex'], 'Pneumonia': row['Pneumonia']})
            except Exception as e:
                pass
    new_df = pd.DataFrame(new_data)
    new_df.to_csv(LOCAL_CSV_PATH, index=False)
    del unet
    tf.keras.backend.clear_session()
    gc.collect()
    print("\n✅ Đã cắt xong và giải phóng bộ nhớ U-Net!")

# ==========================================================
# PHẦN 2: CỖ MÁY GENERATOR & MODEL
# ==========================================================
class FastMultimodalGenerator(Sequence):
    def __init__(self, dataframe, batch_size=32, target_size=(224, 224), shuffle=True):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]
        X_images = np.empty((len(batch_df), *self.target_size, 3), dtype=np.float32)
        X_clinicals = np.empty((len(batch_df), 2), dtype=np.float32)
        Y_labels = np.empty((len(batch_df), 1), dtype=np.float32)
        for i, (_, row) in enumerate(batch_df.iterrows()):
            img = cv2.imread(row['Masked_Path'])
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_resized = cv2.resize(img_rgb, self.target_size)
                img_preprocessed = tf.keras.applications.densenet.preprocess_input(img_resized.astype(np.float32))
            else:
                img_preprocessed = np.zeros((*self.target_size, 3), dtype=np.float32)
            X_images[i] = img_preprocessed
            X_clinicals[i] = [row['Age_norm'], row['Sex']]
            Y_labels[i] = row['Pneumonia']
        return (X_images, X_clinicals), Y_labels

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

def build_fusion_model(densenet_path):
    # 1. NHÁNH LÂM SÀNG (Giữ nguyên)
    clinical_input = Input(shape=(2,), name="clinical_input")
    x_clin = Dense(64, name="clin_dense_1")(clinical_input)
    x_clin = BatchNormalization(name="clin_bn_1")(x_clin)
    x_clin = LeakyReLU(alpha=0.1, name="clin_leaky_1")(x_clin)
    clinical_features = Dense(32, activation='relu', name="clinical_features")(x_clin)

    # 2. NHÁNH HÌNH ẢNH (DENSENET)
    base_cnn = load_model(densenet_path)
    base_cnn.trainable = False

    # ---------------------------------------------------
    # BƯỚC PHẪU THUẬT: SỬA LỖI CỔ CHAI SIGMOID
    # ---------------------------------------------------
    image_input = base_cnn.input

    # Lấy output từ lớp Convolution cuối cùng thay vì lớp Sigmoid cuối cùng
    # Lớp 'conv5_block16_concat' của DenseNet121 chứa chính xác 1024 channels không gian (spatial features)
    x_img_features = base_cnn.get_layer('conv5_block16_concat').output

    # Ép 1024 channels (có dạng ma trận không gian 7x7) thành véc-tơ 1024 chiều bằng GAP
    image_features = GlobalAveragePooling2D(name="fusion_gap")(x_img_features)
    # ---------------------------------------------------

    # 3. HỢP NHẤT VÀ PHÂN LOẠI
    # Trộn 1024 đặc trưng (ảnh) + 32 đặc trưng (lâm sàng) = Véc-tơ 1056 chiều
    fused = Concatenate(name="fusion_concat")([image_features, clinical_features])

    x = Dense(128, activation='relu', name="fusion_dense_last_1")(fused)
    x = Dropout(0.3, name="fusion_dropout_last")(x)
    x = Dense(64, activation='relu', name="fusion_dense_last_2")(x)

    # Lớp Sigmoid MỚI (Dành riêng cho kết luận Viêm phổi của mô hình Fusion)
    output = Dense(1, activation='sigmoid', name="final_prediction")(x)

    model = Model(inputs=[image_input, clinical_input], outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    return model

# ==========================================================
# TRUNG TÂM ĐIỀU KHIỂN (TRAIN & ĐÁNH GIÁ)
# ==========================================================
if __name__ == "__main__":
    # --- BƯỚC A: TẢI DATA ---
    if not os.path.exists(LOCAL_CSV_PATH):
        download_and_preprocess_data()
    else:
        print("⏭️ Bỏ qua bước cắt phổi vì đã có dữ liệu.")

    # --- BƯỚC B: CHUẨN BỊ GENERATOR ---
    print("\n🧠 [3/3] KHỞI ĐỘNG QUÁ TRÌNH HUẤN LUYỆN FUSION MODEL...")
    df = pd.read_csv(LOCAL_CSV_PATH)
    df['Age'] = df['Age'].fillna(df['Age'].median())
    scaler = StandardScaler()
    df['Age_norm'] = scaler.fit_transform(df[['Age']])
    df['Sex'] = df['Sex'].map({'Male': 0, 'Female': 1}).fillna(-1)

    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Pneumonia'])
    train_gen = FastMultimodalGenerator(train_df, batch_size=32, target_size=(224, 224), shuffle=True)
    val_gen = FastMultimodalGenerator(val_df, batch_size=32, target_size=(224, 224), shuffle=False)

    # --- BƯỚC C: HUẤN LUYỆN (CÓ NHẬT KÝ VÀ TRỌNG SỐ PHẠT) ---
    model = build_fusion_model(DENSENET_PATH)
    early_stop = EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True)
    csv_logger = CSVLogger(HISTORY_CSV_PATH, append=False)

    # 1. Tính toán Class Weights dựa trên nhãn của tập Train
    y_train = train_df['Pneumonia'].values
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights_dict = dict(zip(classes, weights))

    print(f"\n⚖️ TRỌNG SỐ PHẠT ĐÃ TÍNH:")
    print(f"   - NORMAL (0): {class_weights_dict[0]:.2f}")
    print(f"   - PNEUMONIA (1): {class_weights_dict[1]:.2f}")

    # 2. Truyền class_weight vào hàm fit
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=30,
        class_weight=class_weights_dict, # <-- ĐÂY LÀ ĐIỂM CHÌA KHÓA CỨU MÔ HÌNH
        callbacks=[early_stop, csv_logger]
    )

    model.save(NEW_SAVE_PATH)
    print(f"\n🎉 ĐÃ LƯU MODEL TẠI: {NEW_SAVE_PATH}")

    # ==========================================================
    # KHÚC SAU TRAIN: VẼ BIỂU ĐỒ & ĐÁNH GIÁ TỰ ĐỘNG
    # ==========================================================
    print("\n📊 ĐANG XUẤT CÁC BÁO CÁO ĐÁNH GIÁ CHUYÊN MÔN...")

    # --- 1. VẼ BIỂU ĐỒ LỊCH SỬ HỌC TẬP (LOSS & AUC) ---
    hist_df = pd.read_csv(HISTORY_CSV_PATH)
    plt.figure(figsize=(14, 5))

    # Biểu đồ Loss
    plt.subplot(1, 2, 1)
    plt.plot(hist_df['epoch'], hist_df['loss'], label='Train Loss', marker='o')
    plt.plot(hist_df['epoch'], hist_df['val_loss'], label='Val Loss', marker='o')
    plt.title('Lịch sử Sai số (Loss)', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Biểu đồ AUC
    plt.subplot(1, 2, 2)
    plt.plot(hist_df['epoch'], hist_df['auc'], label='Train AUC', marker='s')
    plt.plot(hist_df['epoch'], hist_df['val_auc'], label='Val AUC', marker='s')
    plt.title('Lịch sử AUC (Diện tích dưới đường cong)', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # --- 2. DỰ ĐOÁN ĐỂ VẼ MA TRẬN & ROC ---
    print("\n-> AI đang chấm điểm lại tập Validation để vẽ Ma trận...")
    val_gen.shuffle = False
    val_gen.on_epoch_end()
    y_true = val_gen.df['Pneumonia'].values
    y_pred_prob = model.predict(val_gen, steps=len(val_gen), verbose=0).flatten()
    y_pred_binary = (y_pred_prob > 0.5).astype(int)

    # --- 3. VẼ MA TRẬN NHẦM LẪN ---
    cm = confusion_matrix(y_true, y_pred_binary)
    class_names = ['NORMAL', 'PNEUMONIA']

    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 15})
    plt.title('Ma trận Nhầm lẫn (Confusion Matrix)', fontsize=14, fontweight='bold')
    plt.ylabel('Đáp án Thực tế')
    plt.xlabel('AI Dự đoán')

    # --- 4. VẼ ĐƯỜNG CONG ROC ---
    plt.subplot(1, 2, 2)
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (Tỷ lệ dương tính giả)')
    plt.ylabel('True Positive Rate (Tỷ lệ dương tính thật)')
    plt.title('Đường cong ROC (Receiver Operating Characteristic)', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

    # --- 5. BÁO CÁO DẠNG CHỮ ---
    print("\n--- BÁO CÁO PHÂN LOẠI CHI TIẾT ---")
    print(classification_report(y_true, y_pred_binary, target_names=class_names))

⏭️ Bỏ qua bước cắt phổi vì đã có dữ liệu.

🧠 [3/3] KHỞI ĐỘNG QUÁ TRÌNH HUẤN LUYỆN FUSION MODEL...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()



⚖️ TRỌNG SỐ PHẠT ĐÃ TÍNH:
   - NORMAL (0): 1.58
   - PNEUMONIA (1): 0.73
Epoch 1/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 97s 301ms/step - accuracy: 0.5665 - auc: 0.5891 - loss: 0.8084 - val_accuracy: 0.6216 - val_auc: 0.6519 - val_loss: 0.6521
Epoch 2/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 25s 111ms/step - accuracy: 0.5946 - auc: 0.6310 - loss: 0.6861 - val_accuracy: 0.5758 - val_auc: 0.6672 - val_loss: 0.6879
Epoch 3/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 27s 120ms/step - accuracy: 0.6139 - auc: 0.6552 - loss: 0.6633 - val_accuracy: 0.5792 - val_auc: 0.6731 - val_loss: 0.6886
Epoch 4/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 27s 121ms/step - accuracy: 0.6269 - auc: 0.6726 - loss: 0.6509 - val_accuracy: 0.5950 - val_auc: 0.6808 - val_loss: 0.6706
Epoch 5/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 30s 137ms/step - accuracy: 0.6303 - auc: 0.6793 - loss: 0.6467 - val_accuracy: 0.6086 - val_auc: 0.6866 - val_loss: 0.6607
Epoch 6/30
221/221 ━━━━━━━━━━━━━━━━━━━━ 26s 115ms/step - accuracy: 0.6242 - auc: 0.6815 - loss: 0.6438 - val_ac


🎉 ĐÃ LƯU MODEL TẠI: /content/drive/MyDrive/AI_DATA/multimodal_chexpert_best_V3(12 thang 5).h5

📊 ĐANG XUẤT CÁC BÁO CÁO ĐÁNH GIÁ CHUYÊN MÔN...



-> AI đang chấm điểm lại tập Validation để vẽ Ma trận...



--- BÁO CÁO PHÂN LOẠI CHI TIẾT ---
              precision    recall  f1-score   support

      NORMAL       0.47      0.70      0.56       560
   PNEUMONIA       0.82      0.63      0.71      1208

    accuracy                           0.65      1768
   macro avg       0.64      0.67      0.64      1768
weighted avg       0.71      0.65      0.67      1768

